In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error


In [ ]:
# Task 1
data_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(data_path)

In [ ]:
# Task 2
print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 3
df_food.info()

In [ ]:
# Task 4
df_food.describe()

In [ ]:
 # task 5
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('Delivery time')
plt.ylabel('Minutes')
plt.show()

In [ ]:
# Task 1
df_food.drop("Order_ID", axis=1, inplace=True)


In [ ]:
df_food.shape


In [ ]:
# Task 2: Write your code here:
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
for col in ['Traffic_Level', 'Traffic_Level', 'Weather', 'Time_of_Day']:
    df_food[col] = df_food[col].fillna(df_food[col].mode()[0])

df_food['Courier_Experience_yrs'] = df_food['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].mean())
df_food['Delivery_Time'] = df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mean())

print("Missing values remaining:", df_food.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_food.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
for col in categorical_cols:
    le = LabelEncoder()
    df_food[col] = le.fit_transform(df_food[col].astype(str))

df_food.head()

In [ ]:
# Task 5: Write your code here:
numerical_cols = df_food.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = StandardScaler()
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])
df_food.head()

In [ ]:
# Task 6: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('Delivery time')
plt.ylabel('Minutes')
plt.show()

In [ ]:
df_food.shape
df_food.head()


In [ ]:
# Task 1:
X = df_food.drop("Delivery_Time", axis=1)
y = df_food['Delivery_Time']
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, shuffle=True,)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:

"""
for train_idx, val_idx in kfold.split(X,y):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]"""

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
for fold_idx, (train_index, test_index) in enumerate(kfold.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_fold_pred = model.predict(X_test)

    mae_scores.append(mean_absolute_error(y_test, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.sum():,.2f}")
print(f"Avg of MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_cols=["Traffic_Level",'Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs','Weather','Distance_km']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_fold_pred,)
plt.title('Feature predicted time')
plt.xlabel('minutes')
plt.gca()
plt.show()

In [ ]:
# Task Bonus: Write your code here: